# Photometry + GP

This notebook displays a saved run from the source checkout; run it in that
checkout after installing `pip install -e '.[examples]'` and running
`python tools/fetch_example_data.py --bootstrap` to prepare checked local atmosphere inputs.

Recover a finite-source microlensing event and stellar variability together.
The physically plausible synthetic source is a **young, active main-sequence
K dwarf**: mass and radius 0.8 solar, distance 6 kpc, a 5000 K photosphere and
4000 K spots. A **0.5-Earth-mass isolated lens** at 4 kpc, moving at
5 mas/year, gives $t_E=1.77$ hours and $\rho=0.614$; $u_0/\rho=0.7$
makes the lens cross the source.

[CK04 atmospheres](https://etc.stsci.edu/etcstatic/users_guide/1_ref_4_ck.html)
and the [Roman F146 response](https://science.nasa.gov/mission/roman-space-telescope/wfi-technical/)
give source magnitude 20.79 AB and blended baseline 20.55 AB, assuming
1 mag extinction at 1.65 microns and 80% source light. Mean projected spot
coverage 10%, with 2 percentage-point RMS variation, gives **0.82% intrinsic
F146 flux RMS**. The 5-day rotation and 20-day coherence are scenario choices,
motivated by [young-star observations](https://arxiv.org/abs/1606.00057).

An idealized gapless 70.5-day season uses 12.1-minute cadence and 66.4-second
exposures. Known Gaussian errors approximate the
[Roman survey sensitivity](https://roman-docs.stsci.edu/roman-community-defined-surveys/galactic-bulge-time-domain-survey),
plus an assumed independent 1 mmag floor. Total peak brightness is about
19.49 AB, fainter than the conservative 19.2 AB saturation guard.
This is a synthetic inference example, not a crowded-field instrument simulation.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))


In [ ]:
from IPython.display import Image, Markdown, display
from example.inference.code.active_plots import OUTPUT, load_results

experiment = load_results()  # verifies the completed run and saved products
report = experiment["report"]
sampling = report["sampling"]
rho = report["posterior"]["rho"]
comparison = report["physical_checks"]["finite_source"]["comparison"]
map_delta = comparison["actual_data_gp_pspl"]["delta_2_log_likelihood_at_map"]
display(Markdown(
    f"**Recovered ρ = {rho['median']:.3f}** "
    f"(68% interval {rho['q16']:.3f}–{rho['q84']:.3f}; "
    f"injected {report['truth']['rho']:.3f}). "
    f"Peak **F146 = {report['peak_mag_ab']:.2f} AB**.\n\n"
    f"Simulated-data finite-source versus point-source + GP: "
    f"Δ2 log L at the separate MAPs = {map_delta:.1f}; "
    "this is not a calibrated detection significance.\n\n"
    f"Convergence: max R̂ {sampling['max_r_hat']:.3f}, "
    f"min ESS {sampling['min_n_eff']:.0f}, "
    f"divergences {sampling['divergences']}."
))

## Fit the event and variability together

For magnification $A(t)$ and intrinsic source-flux variation $g(t)$,

$$F_i=A_i[f_s+g_i]+f_b+\epsilon_i,\qquad
C_{ij}=A_iA_jK_{ij}+\delta_{ij}\sigma_i^2.$$

The fit uses `Model(source="linear", solver="fft")`, three-point exposure
averaging, and fixed $a_1=0.4$. Two damped oscillators describe rotation and
its first harmonic with an 80:20 variance split. tinygp integrates out the
latent GP; no jitter is fitted.

### Likelihood and priors

The implementation evaluates the Gaussian density in intrinsic source-flux
units. The final term converts its normalization back to observed flux:

```{literalinclude} ../../example/inference/code/active_ffp.py
:language: python
:start-after: BEGIN ACTIVE GP LIKELIHOOD
:end-before: END ACTIVE GP LIKELIHOOD
```

All nine parameters are inferred jointly. Priors are independent Normals in
the coordinates below; logs are natural and times are in days.
Write $b=f_s+f_b$, $s=f_s/b$, and $q_k=(c_k-\mu_k)/\sigma_k$.

| Coordinate $c_k$ | Mean $\mu_k$ | Standard deviation $\sigma_k$ |
| --- | --- | --- |
| $t_0$ | 0 | 0.2 |
| $\log u_0$ | $\log 0.4$ | 0.7 |
| $\log t_E$ | $\log 0.1$ | 0.7 |
| $\log b$ | 0 | 0.05 |
| $\operatorname{logit}s$ | $\operatorname{logit}0.75$ | 1.2 |
| $\log \sigma_{\rm GP}$, fractional source RMS | $\log 0.01$ | 1.0 |
| $\log \tau_{\rm GP}$, coherence time | $\log 15$ | 0.8 |
| $\log P_{\rm GP}$, rotation period | $\log 6$ | 0.7 |
| $\log\rho$ | $\log 0.6$ | 0.8 |

### NumPyro model

`sample()` uses source-crossing coordinates $y$ and a fixed transformation
from the MAP curvature to reduce posterior correlations. Its objective
includes the marginal likelihood, standardized Normal prior, and coordinate
Jacobian:

```{literalinclude} ../../example/inference/code/active_ffp.py
:language: python
:start-after: BEGIN ACTIVE OBJECTIVE
:end-before: END ACTIVE OBJECTIVE
:dedent: 4
```

The NumPyro model below cancels the auxiliary standard-Normal density of
`z`, so the physical priors remain those in the table:

```{literalinclude} ../../example/inference/code/active_ffp.py
:language: python
:start-after: BEGIN ACTIVE NUMPYRO
:end-before: END ACTIVE NUMPYRO
:dedent: 4
```

The displayed run uses two sequential chains, 600 warmup steps and 1600
retained draws per chain, with target acceptance 0.9. The
{download}`full script <../../example/inference/code/active_ffp.py>` contains
the GP kernel, exposure integration, MAP initialization, and `mcmc.run()`;
{download}`coordinate transforms <../../example/inference/code/active_coordinates.py>`
define $q(y)$ and its Jacobian.

The scalar-GP approximation has a
{download}`frozen-spot check <../../example/inference/outputs/active_ffp/spatial_spot_check.pdf>`
with noiseless data and known local variability. Nearby spots can bias timing
even without a crossing; this check does not validate arbitrary GP realizations.
Annual parallax is checked separately. Neither lens mass nor the absence of
a host is measured: finite-source structure alone does not provide $\pi_E$.


## Inspect the recovery

The full-season figure shows F146 magnitudes, variability relative to the
**injected lens-only curve**, and full-model residuals. The event zoom compares
the finite-source fit with a **refitted point-source + GP model**; its trajectory,
blending and GP parameters are allowed to change. The corner plot shows the
nine-parameter posterior and injected values.

Blue bands contain 68%/95% of latent predictions, including parameter and
conditional-GP uncertainty, without new measurement noise. Display bins show
measurement-noise errors only; all observations enter the fit. One realization
does not establish coverage.

In [ ]:
display(Image(filename=str(OUTPUT / "recovery.png")))

In [ ]:
display(Image(filename=str(OUTPUT / "event_zoom.png")))

In [ ]:
display(Image(filename=str(OUTPUT / "corner.png")))


## Reproduce

Install the example dependencies and run from the repository root:

```bash
python -m pip install -e '.[examples]'
python tools/fetch_example_data.py --bootstrap
python -m example.inference.code.active_ffp --warmup 600 --samples 1600 --progress
python -m example.inference.code.active_plots
python -m example.inference.code.active_spots
```

The data, posterior, conditional predictions, physical assumptions and
convergence diagnostics live in `example/inference/outputs/active_ffp/`.
The loader rejects incomplete runs or mismatched files. Download the
{download}`recovery PDF <../../example/inference/outputs/active_ffp/recovery.pdf>`,
{download}`event-zoom PDF <../../example/inference/outputs/active_ffp/event_zoom.pdf>`
and {download}`corner PDF <../../example/inference/outputs/active_ffp/corner.pdf>`.
The [inference benchmark](../benchmarks.md#inference) compares samplers
for a separate stellar-lens event with annual parallax.
